# 股票分析器（Stock Analyzer）

## 练习目标（理念）

从 **nairametrics.com** 抓取一篇尼日利亚公司新闻，再交给 **LLM** 分析，向新手投资者给出 **买入 / 卖出 / 持有（Buy / Sell / Hold）** 建议。

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 + 清洗 | `requests` + `BeautifulSoup` 取正文 |
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定分析师角色；user 放新闻正文 |
| 结果展示 | `display(Markdown(...))` |

## 怎么跑

1. 准备好 `.env`（含 `OPENAI_API_KEY`），从上到下依次运行单元格
2. 需要本地模型时，取消注释 Ollama 的 `base_url` / `model` 行
3. 在最后一格改新闻 URL，再跑 `display_analysis(...)`


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 requests：用 HTTP GET 抓取网页 HTML
import requests
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：在笔记本里漂亮地渲染 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端（或 Ollama 兼容）Chat Completions API
from openai import OpenAI
# 从 bs4 导入 BeautifulSoup：解析 HTML，定位文章正文区域
from bs4 import BeautifulSoup
# 再次导入 requests（原作者重复导入；保持原样，不影响行为）
import requests


In [ ]:
# ========== 抓取：用固定请求头 + BeautifulSoup 抽正文 ==========

# 抓取网页用的标准请求头（User-Agent）：模拟浏览器，降低被站点拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    # 入参 url：要分析的新闻页地址；返回文章正文纯文本（或找不到时的英文占位串）
    # 发送 GET 请求；headers 带上浏览器身份
    response = requests.get(url, headers=headers)
    # 用 BeautifulSoup 解析响应体（response.content 是字节）
    soup = BeautifulSoup(response.content, "html.parser")
    # print(soup.prettify())  # 调试时可打开：查看整页 HTML 结构
    # 按站点结构定位正文容器：div.content-inner.jeg_link_underline（Nairametrics 特有）
    content = soup.find('div', class_='content-inner jeg_link_underline')
    # 找到则取纯文本；否则返回英文占位（影响下游 prompt，勿改译）
    return content.get_text() if content else "Content not found"



In [ ]:
# ========== system prompt：告诉模型「你是谁、怎么答」==========

# 资深股票分析师角色提示（发给模型的指令字符串，保留英文，改译会改变回答风格）
system_prompt = """
You are seasoned stock analyst and speculator.
You are very good at analyzing news articles of a Nigerian company published on Nairametrics,
with the information in the article, you are able to give solid advice to a completely novice investor on whether to buy, sell or hold the stock of the company in question.

You always give your advice in this format:
# Company Name
## Summary of the news article in 2-3 sentences
## Advice to potential investors (Buy, Sell or Hold)
## Reasoning for the advice.

You always tailor your advice to suit novice investors only, giving a detailed explanation of the reasoning behind your advice.
Avoid using a lot of technical jargon, and if you must use technical terms, always explain them in simple terms.
"""



In [ ]:
# ========== user prompt 前缀：用户消息的固定开头 ==========

# 用户提示前缀：后面会拼上抓取到的新闻正文（字符串本身保留英文）
user_prompt_prefix = """
Here are the contents of a company news article.
Provide an analysis of the news article.

"""



In [ ]:
# ========== messages_for：把 system + user 打成 API 需要的列表 ==========

def messages_for(website):
    # website：上一格抓到的正文文本
    # 组装系统 + 用户消息：Chat Completions 标准格式 role/content
    return [
        {"role": "system", "content": system_prompt},
        # user 内容 = 前缀说明 + 新闻正文（字符串拼接）
        {"role": "user", "content": user_prompt_prefix + website}
    ]



In [ ]:
# ========== 客户端：默认云端 OpenAI；可选切到本地 Ollama ==========

# Ollama 的 OpenAI 兼容端点（/v1）：本地服务地址，取消下面注释即可改用
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建默认 OpenAI 客户端：密钥通常来自环境变量 OPENAI_API_KEY（需先 load_dotenv）
openai = OpenAI()
# 也可改用本地 Ollama：把 base_url 指到本机，api_key 任意非空即可（Ollama 常不校验）
# openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')



In [ ]:
# ========== analyze_article：抓取 → 组消息 → 调模型 → 取回答 ==========

def analyze_article(url):
    # 抓取文章正文（失败时可能是 "Content not found"）
    website = fetch_website_contents(url)
    # 调用 Chat Completions：model / messages 决定用谁、问什么
    response = openai.chat.completions.create(
        # 云端小模型：便宜、够用；字符串是 model id，勿随意改译
        model = "gpt-4.1-mini",
        # model = "llama3.2:1b",  # 若上面客户端切到 Ollama，可改用这些本地模型名
        # model = "gemma3:270m",
        # messages：system 定角色 + user 放新闻
        messages = messages_for(website)
    )
    # 取第一条 choice 的 message.content（模型最终文本）
    return response.choices[0].message.content



In [ ]:
# ========== display_analysis：分析结果以 Markdown 渲染 ==========

def display_analysis(url):
    # 先跑完整分析流水线，得到 Markdown 文本
    analysis = analyze_article(url)
    # 在 Jupyter 里渲染 Markdown（标题层级、列表会好看很多）
    display(Markdown(analysis))



In [ ]:
# ========== 入口：换 URL 即可分析另一篇新闻 ==========

# 调用展示函数；参数是 Nairametrics 上的具体文章链接（影响抓取内容，保持原样）
display_analysis("https://nairametrics.com/2026/04/23/unilever-nigeria-plc-sustains-double-digit-growth-momentum-in-q1-2026-unaudited-results/")
